JSON EXAMPLE ZERO TO HERO

{"id":1, "name":"Somesh", "skills":["spark","python"]}


STEP 1 — Create DataFrame

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

spark = SparkSession.builder.getOrCreate()

data = [
    ('{"id":1, "name":"Somesh", "skills":["spark","python"]}',),
    ('{"id":2, "name":"Anita", "skills":["excel","hcm"]}',),
    ('{"id":3, "name":"Karan", "skills":["sql","finance"]}',)
]

df = spark.createDataFrame(data, ["json_col"])
df.show(truncate=False)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/23 14:08:28 WARN Utils: Your hostname, codespaces-b99eb4, resolves to a loopback address: 127.0.0.1; using 10.0.1.96 instead (on interface eth0)
25/11/23 14:08:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/23 14:08:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/23 14:08:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+------------------------------------------------------+
|json_col                                              |
+------------------------------------------------------+
|{"id":1, "name":"Somesh", "skills":["spark","python"]}|
|{"id":2, "name":"Anita", "skills":["excel","hcm"]}    |
|{"id":3, "name":"Karan", "skills":["sql","finance"]}  |
+------------------------------------------------------+



STEP 2 — Define schema

In [2]:
schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("skills", ArrayType(StringType()))
])


STEP 3 — Parse JSON

In [3]:
df2 = df.withColumn("json", from_json(col("json_col"), schema))
df2.show(truncate=False)


+------------------------------------------------------+----------------------------+
|json_col                                              |json                        |
+------------------------------------------------------+----------------------------+
|{"id":1, "name":"Somesh", "skills":["spark","python"]}|{1, Somesh, [spark, python]}|
|{"id":2, "name":"Anita", "skills":["excel","hcm"]}    |{2, Anita, [excel, hcm]}    |
|{"id":3, "name":"Karan", "skills":["sql","finance"]}  |{3, Karan, [sql, finance]}  |
+------------------------------------------------------+----------------------------+



STEP 4 — Expand JSON

In [6]:
df3 = df2.select("json.*")
df3.show(truncate=False)


+---+------+---------------+
|id |name  |skills         |
+---+------+---------------+
|1  |Somesh|[spark, python]|
|2  |Anita |[excel, hcm]   |
|3  |Karan |[sql, finance] |
+---+------+---------------+



STEP 5 — Explode the array into multiple rows

In [7]:
from pyspark.sql.functions import explode

df4 = df3.select("id", "name", explode("skills").alias("skill"))
df4.show()


+---+------+-------+
| id|  name|  skill|
+---+------+-------+
|  1|Somesh|  spark|
|  1|Somesh| python|
|  2| Anita|  excel|
|  2| Anita|    hcm|
|  3| Karan|    sql|
|  3| Karan|finance|
+---+------+-------+

